In [1]:
print(12)

12


In [2]:
from kafka import KafkaConsumer

server = 'localhost:9092'
topic_name = 'rides'

In [3]:
from models import Ride,  ride_deserializer

In [ ]:
consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='rides-database',
    value_deserializer=lambda x:x
)

In [5]:
import psycopg2

conn = psycopg2.connect(
    host='localhost',
    port=5432,
    database='postgres',
    user='postgres',
    password='postgres'
)
conn.autocommit = True
cur = conn.cursor()

In [ ]:
# import json
# for record in consumer:
#     print(record.value)
#     print(json.loads(record.value.decode('utf-8')).get('tpep_pickup_datetime'))

b'{"PULocationID": 43, "DOLocationID": 186, "trip_distance": 1.68, "total_amount": 22.15, "tpep_pickup_datetime": 1761956005000}'
1761956005000
b'{"PULocationID": 142, "DOLocationID": 237, "trip_distance": 2.28, "total_amount": 24.94, "tpep_pickup_datetime": 1761958147000}'
1761958147000
b'{"PULocationID": 163, "DOLocationID": 238, "trip_distance": 2.7, "total_amount": 25.62, "tpep_pickup_datetime": 1761955639000}'
1761955639000
b'{"PULocationID": 138, "DOLocationID": 261, "trip_distance": 12.87, "total_amount": 86.14, "tpep_pickup_datetime": 1761955200000}'
1761955200000
b'{"PULocationID": 138, "DOLocationID": 37, "trip_distance": 8.4, "total_amount": 48.65, "tpep_pickup_datetime": 1761956330000}'
1761956330000
b'{"PULocationID": 90, "DOLocationID": 100, "trip_distance": 0.85, "total_amount": 16.45, "tpep_pickup_datetime": 1761956471000}'
1761956471000
b'{"PULocationID": 142, "DOLocationID": 170, "trip_distance": 3.01, "total_amount": 25.85, "tpep_pickup_datetime": 1761955651000}'
176

KeyboardInterrupt: 

In [22]:
from datetime import datetime

print(f"Listening to {topic_name} and writing to PostgreSQL...")

count = 0
for message in consumer:
    # AA: I invoked the deserializer since the sent message was bytes 
    ride = ride_deserializer(message.value)
    pickup_dt = datetime.fromtimestamp(ride.tpep_pickup_datetime / 1000)
    cur.execute(
        """INSERT INTO processed_events
           (PULocationID, DOLocationID, trip_distance, total_amount, pickup_datetime)
           VALUES (%s, %s, %s, %s, %s)""",
        (ride.PULocationID, ride.DOLocationID,
         ride.trip_distance, ride.total_amount, pickup_dt)
    )
    count += 1
    if count % 100 == 0:
        print(f"Inserted {count} rows...")

consumer.close()
cur.close()
conn.close()

Listening to rides and writing to PostgreSQL...
Inserted 100 rows...
Inserted 200 rows...
Inserted 300 rows...
Inserted 400 rows...
Inserted 500 rows...
Inserted 600 rows...
Inserted 700 rows...
Inserted 800 rows...
Inserted 900 rows...
Inserted 1000 rows...


KeyboardInterrupt: 